# DoG threshold sweep, scored with the competition metric

Sweeps `DOG_THR_PCT` through the submission notebook's own detect -> mask -> Trackastra
pipeline on train movies, scoring each with a re-implementation of the organisers'
metric (after sleepymegacat/the-metric-decides-your-architecture-8-measured):
node-count-adjusted edge Jaccard. Division Jaccard (0.1 weight) is not scored.

The visible test/ volumes are copies of train samples, so they are excluded here.


In [ ]:
# Step 2 — install only Trackastra; keep Kaggle's already-loaded numeric stack intact.
_WHEELS = "/kaggle/input/models/jirkaborovec/biohubcelltrack-trackastra-artifact-and-packages/pytorch/default/1/packages/packages"
! pip install -q --no-index --find-links "{_WHEELS}" --no-deps trackastra==0.5.3 lz4 edt imagecodecs chardet dask geff geff-spec joblib pyyaml tqdm requests psutil platformdirs networkx cloudpickle fsspec partd locket toolz zarr==3.2.1 numcodecs==0.15.1 donfig google-crc32c
! python -c "from trackastra.model import Trackastra; import trackastra, numpy, scipy; print('offline model import OK', trackastra.__version__, numpy.__version__, scipy.__version__)"

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import logging
import os
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max
from skimage.morphology import ball, binary_dilation
from skimage.segmentation import watershed
from tqdm.auto import tqdm

logging.getLogger("trackastra").setLevel(logging.WARNING)

# Paths — override via env vars for local development
DATA_DIR = Path(os.environ.get("KAGGLE_DATA_DIR", "/kaggle/input/competitions/biohub-cell-tracking-during-development"))
OUTPUT_DIR = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "."))  # "." = /kaggle/working/ on Kaggle
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Train exists: {TRAIN_DIR.exists()}, Test exists: {TEST_DIR.exists()}")

In [ ]:
def get_volume_shape(zarr_path: Path) -> tuple[int, int, int, int]:
    """Return (T, Z, Y, X) from zarr.json metadata — no zarr import needed."""
    with (zarr_path / "0" / "zarr.json").open() as f:
        return tuple(json.load(f)["shape"])


def load_timepoint_blosc2(zarr_path: Path, t: int) -> np.ndarray:
    """Load one timepoint (Z, Y, X) directly from blosc2 chunk.

    Chunk path: <zarr>/0/c/<t>/0/0/0. Faster than zarr array slicing.
    """
    import blosc2

    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    raw = chunk_path.read_bytes()
    return np.frombuffer(blosc2.decompress(raw), dtype=np.uint16).reshape(-1)


def load_timepoint(zarr_path: Path, t: int, shape_zyx: tuple[int, int, int]) -> np.ndarray:
    """Load one timepoint with blosc2 fast path and zarr fallback."""
    try:
        flat = load_timepoint_blosc2(zarr_path, t)
        n = shape_zyx[0] * shape_zyx[1] * shape_zyx[2]
        return flat[:n].reshape(shape_zyx).astype(np.float32)
    except Exception:
        import zarr as _zarr
        return _zarr.open(str(zarr_path), mode="r")["0"][t].astype(np.float32)

In [ ]:
# Physical voxel scale (µm/voxel)
VOXEL_Z = 1.625
VOXEL_Y = 0.40625
VOXEL_X = 0.40625
XY_DOWNSAMPLE = 4          # block-mean XY to get ~isotropic 1.625 µm grid
SCALE_ZYX = np.array([VOXEL_Z, VOXEL_Y, VOXEL_X])


def make_isotropic(vol: np.ndarray) -> np.ndarray:
    """Block-average XY by XY_DOWNSAMPLE to get isotropic voxels.

    Raw: z=1.625, y=x=0.406 µm. After ÷4: z=y=x=1.625 µm.
    """
    nz, ny, nx = vol.shape
    ny2 = ny // XY_DOWNSAMPLE
    nx2 = nx // XY_DOWNSAMPLE
    cropped = vol[:, : ny2 * XY_DOWNSAMPLE, : nx2 * XY_DOWNSAMPLE]
    return cropped.reshape(nz, ny2, XY_DOWNSAMPLE, nx2, XY_DOWNSAMPLE).mean(axis=(2, 4))

In [ ]:
DOG_SIGMAS = (1.0, 1.8, 3.0)
DOG_K = 1.6          # sigma_high = sigma_low × DOG_K
DOG_THR_PCT = 80.0   # keep peaks above this percentile of positive DoG response
REFINE_RZ = 2        # COM half-window in Z (original voxels)
REFINE_RYX = 5       # COM half-window in Y/X (original voxels)
MIN_PEAK_DIST = 2    # min peak separation in isotropic voxels
NMS_RADIUS_UM = 4.0  # physical dedup radius (µm) — ~half typical cell diameter


def _dog_scale_back(pk_iso: np.ndarray) -> np.ndarray:
    """Convert isotropic peak coords to original-voxel float coords.

    Places y, x at the block centre (not edge).
    """
    out = pk_iso.astype(float)
    out[:, 1] = out[:, 1] * XY_DOWNSAMPLE + (XY_DOWNSAMPLE - 1) / 2.0
    out[:, 2] = out[:, 2] * XY_DOWNSAMPLE + (XY_DOWNSAMPLE - 1) / 2.0
    return out


def _com_refine_orig(vol: np.ndarray, zyx: np.ndarray) -> np.ndarray:
    """Intensity-weighted COM refinement in original anisotropic volume.

    Asymmetric window: REFINE_RZ in Z, REFINE_RYX in Y/X (~spherical in µm).
    """
    nz, ny, nx = vol.shape
    z, y, x = int(round(zyx[0])), int(round(zyx[1])), int(round(zyx[2]))
    z0, z1 = max(0, z - REFINE_RZ), min(nz, z + REFINE_RZ + 1)
    y0, y1 = max(0, y - REFINE_RYX), min(ny, y + REFINE_RYX + 1)
    x0, x1 = max(0, x - REFINE_RYX), min(nx, x + REFINE_RYX + 1)
    patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
    w = np.maximum(patch - float(patch.min()), 0.0)
    total = float(w.sum())
    if total < 1e-12:
        return zyx.copy()
    gz, gy, gx = np.mgrid[z0:z1, y0:y1, x0:x1]
    return np.array([(gz * w).sum(), (gy * w).sum(), (gx * w).sum()]) / total


def _nms_physical(
    coords: np.ndarray, scores: np.ndarray, radius_um: float
) -> tuple[np.ndarray, np.ndarray]:
    """Non-maximum suppression in physical µm space.

    Suppresses all lower-score peaks within radius_um of a stronger peak.
    """
    if len(coords) <= 1:
        return coords, scores
    pts = coords * SCALE_ZYX[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), dtype=bool)
    keep: list[int] = []
    for i in order:
        if killed[i]:
            continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True
    k = np.array(keep)
    return coords[k], scores[k]


def detect_peaks_dog(
    vol: np.ndarray,
    dog_thr_pct: float = DOG_THR_PCT,
    topk: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Multi-scale DoG detection on raw anisotropic volume.

    Returns:
        coords: (N, 3) float array in original voxel space (z, y, x).
        scores: (N,) normalised DoG response scores.
    """
    pooled = make_isotropic(vol)
    all_coords: list[np.ndarray] = []
    all_scores: list[float] = []

    for sigma in DOG_SIGMAS:
        dog = gaussian_filter(pooled, sigma) - gaussian_filter(pooled, sigma * DOG_K)
        pos_vals = dog[dog > 0]
        if pos_vals.size == 0:
            continue
        thr = float(np.percentile(pos_vals, dog_thr_pct))
        iso_peaks = peak_local_max(
            dog, min_distance=MIN_PEAK_DIST, threshold_abs=thr, exclude_border=False
        )
        if len(iso_peaks) == 0:
            continue
        resp = dog[iso_peaks[:, 0], iso_peaks[:, 1], iso_peaks[:, 2]].astype(float)
        resp /= max(float(resp.max()), 1e-6)
        orig_init = _dog_scale_back(iso_peaks)
        for p, r in zip(orig_init, resp, strict=False):
            all_coords.append(_com_refine_orig(vol, p))
            all_scores.append(float(r))

    if not all_coords:
        return np.zeros((0, 3), dtype=float), np.zeros(0, dtype=float)

    coords = np.array(all_coords)
    scores = np.array(all_scores)
    coords, scores = _nms_physical(coords, scores, NMS_RADIUS_UM)

    if topk is not None and len(coords) > topk:
        best = np.argsort(-scores)[: int(topk)]
        coords, scores = coords[best], scores[best]

    return coords, scores

In [ ]:
BALL_RADIUS_ISO = 3       # sphere radius in isotropic voxels (~4.9 µm) — ball mode
MAX_CELL_RADIUS_ISO = 5   # watershed basin cap in isotropic voxels (~8 µm — one cell)
SEG_FG_PCT = 75.0         # intensity percentile: watershed grows only into brighter voxels


def centroids_to_mask(
    shape_zyx_iso: tuple[int, int, int],
    centroids_orig: list[tuple[float, float, float]],
    radius: int = BALL_RADIUS_ISO,
) -> np.ndarray:
    """Create uint16 ball-instance mask from original-voxel centroid list.

    Args:
        shape_zyx_iso: (Z, Y_iso, X_iso) isotropic volume shape.
        centroids_orig: List of (z, y, x) float coords in original voxel space.
        radius: Ball radius in isotropic voxels.

    Returns:
        uint16 mask (Z, Y_iso, X_iso); background=0, cells=1,2,...,N.
    """
    nz, ny, nx = shape_zyx_iso
    mask = np.zeros((nz, ny, nx), dtype=np.uint16)

    for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
        zi = int(round(z_o))
        yi = int(round(y_o / XY_DOWNSAMPLE))
        xi = int(round(x_o / XY_DOWNSAMPLE))

        z0, z1 = max(0, zi - radius), min(nz, zi + radius + 1)
        y0, y1 = max(0, yi - radius), min(ny, yi + radius + 1)
        x0, x1 = max(0, xi - radius), min(nx, xi + radius + 1)

        gz, gy, gx = np.ogrid[z0:z1, y0:y1, x0:x1]
        sphere = ((gz - zi) ** 2 + (gy - yi) ** 2 + (gx - xi) ** 2) <= radius ** 2
        patch = mask[z0:z1, y0:y1, x0:x1]
        patch[sphere & (patch == 0)] = np.uint16(cell_id)

    return mask


def centroids_to_seg_mask(
    vol_iso: np.ndarray,
    centroids_orig: list[tuple[float, float, float]],
    max_radius: int = MAX_CELL_RADIUS_ISO,
    fg_pct: float = SEG_FG_PCT,
) -> np.ndarray:
    """Marker-controlled watershed instance mask seeded by DoG centroids.

    Grows a real cell-shaped region per centroid on the isotropic intensity volume,
    giving Trackastra genuine appearance/morphology features (unlike fixed balls).
    Each basin is bounded to a sphere of ``max_radius`` around its seed and to voxels
    brighter than the ``fg_pct`` intensity percentile, so a bright neighbour cannot
    swallow the frame.

    Args:
        vol_iso: Isotropic intensity volume (Z, Y_iso, X_iso) — same array fed to the tracker.
        centroids_orig: List of (z, y, x) float coords in original voxel space.
        max_radius: Basin radius cap in isotropic voxels.
        fg_pct: Intensity percentile; watershed grows only into voxels above it.

    Returns:
        uint16 mask (Z, Y_iso, X_iso); background=0, cells=1,2,...,N.
    """
    nz, ny, nx = vol_iso.shape
    markers = np.zeros((nz, ny, nx), dtype=np.int32)
    for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
        zi = min(max(int(round(z_o)), 0), nz - 1)
        yi = min(max(int(round(y_o / XY_DOWNSAMPLE)), 0), ny - 1)
        xi = min(max(int(round(x_o / XY_DOWNSAMPLE)), 0), nx - 1)
        markers[zi, yi, xi] = cell_id

    if not centroids_orig:
        return markers.astype(np.uint16)

    seed_mask = markers > 0
    bound = binary_dilation(seed_mask, ball(max_radius))         # spatial cap per cell
    fg = ((vol_iso > np.percentile(vol_iso, fg_pct)) & bound) | seed_mask
    labels = watershed(-vol_iso, markers, mask=fg, compactness=0.01)
    return labels.astype(np.uint16)

In [ ]:
from trackastra.model import Trackastra

MODEL_DIR = Path("/kaggle/input/models/jirkaborovec/biohubcelltrack-trackastra-artifact-and-packages/pytorch/default/1/models/ctc")  # bundled weights from Setup Step 3

if MODEL_DIR.exists():
    model = Trackastra.from_folder(MODEL_DIR, device=DEVICE)
    print(f"Loaded Trackastra from {MODEL_DIR}")
else:
    # Online fallback — only works during dev; internet disabled on Kaggle submission
    model = Trackastra.from_pretrained("ctc", device=DEVICE)
    print("Loaded Trackastra from pretrained hub (online)")

In [ ]:
MASK_MODE = "watershed"  # "watershed" (real masks, appearance features) or "ball" (fixed spheres)
PRUNE_ISOLATED = True    # drop zero-degree nodes — almost all are false positives
TOPK_PER_FRAME = None    # cap detections/frame by DoG score (None = keep all)
TRACK_MODE = "greedy"    # "greedy" (fast) or "ilp" (global, better, slower)


@dataclass(frozen=True, slots=True)
class NodeRow:
    """One detected cell centroid at one timepoint."""

    dataset: str
    node_id: int
    t: int
    z: int
    y: int
    x: int


@dataclass(frozen=True, slots=True)
class EdgeRow:
    """Temporal link between two detections."""

    dataset: str
    source_id: int
    target_id: int


def track_one_dataset(
    zarr_path: Path,
    id_offset: int = 0,
    topk: int | None = TOPK_PER_FRAME,
    mode: str = TRACK_MODE,
) -> tuple[list[NodeRow], list[EdgeRow], int]:
    """Detect → mask → Trackastra track → prune, for one zarr volume.

    Node IDs are assigned as ``id_offset + 1, id_offset + 2, ...`` over *all* graph
    nodes (before pruning), so IDs never collide across datasets even after pruning
    removes some of them.

    Args:
        zarr_path: Path to the .zarr volume.
        id_offset: Add to every local node ID for global uniqueness.
        topk: Optional per-frame detection cap by DoG score.
        mode: Trackastra tracking mode ("greedy" or "ilp").

    Returns:
        (nodes, edges, n_assigned) — n_assigned is the count of IDs consumed
        (pre-prune), which the caller adds to its running offset.
    """
    dataset = zarr_path.stem
    n_t, n_z, n_y, n_x = get_volume_shape(zarr_path)
    n_y_iso = n_y // XY_DOWNSAMPLE
    n_x_iso = n_x // XY_DOWNSAMPLE
    shape_zyx = (n_z, n_y, n_x)
    shape_iso = (n_z, n_y_iso, n_x_iso)

    det_lookup: dict[tuple[int, int], tuple[float, float, float]] = {}
    imgs_4d = np.zeros((n_t, n_z, n_y_iso, n_x_iso), dtype=np.float32)
    masks_4d = np.zeros((n_t, n_z, n_y_iso, n_x_iso), dtype=np.uint16)

    for t in tqdm(range(n_t), desc=f"  {dataset} frames", leave=False):
        vol = load_timepoint(zarr_path, t, shape_zyx)
        coords, _scores = detect_peaks_dog(vol, topk=topk)
        imgs_4d[t] = make_isotropic(vol)
        centroids_orig = [(float(c[0]), float(c[1]), float(c[2])) for c in coords]
        if MASK_MODE == "watershed":
            masks_4d[t] = centroids_to_seg_mask(imgs_4d[t], centroids_orig)
        else:
            masks_4d[t] = centroids_to_mask(shape_iso, centroids_orig)
        for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
            det_lookup[(t, cell_id)] = (z_o, y_o, x_o)

    with contextlib.redirect_stderr(io.StringIO()):
        track_graph, _masks_tracked = model.track(
            imgs_4d.astype(np.uint16), masks_4d, mode=mode
        )

    node_id_map: dict = {}
    nodes: list[NodeRow] = []
    nid = id_offset + 1
    for graph_node, attrs in track_graph.nodes(data=True):
        centroid = det_lookup.get((int(attrs["time"]), int(attrs["label"])))
        if centroid is None:
            continue  # ghost node not in our mask (shouldn't happen)
        z_o, y_o, x_o = centroid
        node_id_map[graph_node] = nid
        nodes.append(
            NodeRow(dataset, nid, int(attrs["time"]), int(round(z_o)), int(round(y_o)), int(round(x_o)))
        )
        nid += 1
    n_assigned = len(node_id_map)

    edges: list[EdgeRow] = []
    for src, dst in track_graph.edges():
        s, d = node_id_map.get(src), node_id_map.get(dst)
        if s is not None and d is not None:
            edges.append(EdgeRow(dataset, s, d))

    if PRUNE_ISOLATED and edges:
        used = {e.source_id for e in edges} | {e.target_id for e in edges}
        nodes = [n for n in nodes if n.node_id in used]

    return nodes, edges, n_assigned

In [ ]:
MAX_D_UM = 7.0
NODE_ALPHA = 0.1


def load_gt(zarr_path: Path) -> dict:
    import zarr as _zarr

    root = zarr_path.with_suffix(".geff")
    g = _zarr.open(str(root), mode="r")
    ids = np.asarray(g["nodes/ids"])
    row_of = {int(v): i for i, v in enumerate(ids)}
    raw_edges = np.asarray(g["edges/ids"]).reshape(-1, 2)
    edges = np.array([[row_of[int(a)], row_of[int(b)]] for a, b in raw_edges], dtype=np.int64).reshape(-1, 2)
    with (root / "zarr.json").open() as f:
        meta = json.load(f)["attributes"]["geff"]
    return {
        "t": np.asarray(g["nodes/props/t/values"]).astype(np.int64),
        "zyx": np.stack([np.asarray(g[f"nodes/props/{k}/values"]) for k in "zyx"], 1).astype(float),
        "edges": edges,
        "est_total": int((meta.get("extra") or {}).get("estimated_number_of_nodes", 0)),
    }


def match_nodes(p_t, p_um, g_t, g_um):
    """One-to-one per-frame matching maximising sum 1/(1+d) within MAX_D_UM."""
    matched = np.full(len(p_t), -1, np.int64)
    for t in np.intersect1d(np.unique(p_t), np.unique(g_t)):
        pi, gi = np.flatnonzero(p_t == t), np.flatnonzero(g_t == t)
        d = np.sqrt(((g_um[gi][:, None] - p_um[pi][None]) ** 2).sum(2))
        w = np.where(d <= MAX_D_UM, 1.0 / (1.0 + d), -1.0)
        if not (w > -1.0).any():
            continue
        r, c = linear_sum_assignment(w, maximize=True)
        keep = w[r, c] > -1.0
        matched[pi[c[keep]]] = gi[r[keep]]
    return matched


def score_sample(nodes: list[NodeRow], edges: list[EdgeRow], gt: dict) -> dict:
    """Node-count-adjusted edge Jaccard, mirroring the scorer's edge filters."""
    row_of = {n.node_id: i for i, n in enumerate(nodes)}
    p_t = np.array([n.t for n in nodes], dtype=np.int64)
    p_zyx = np.array([(n.z, n.y, n.x) for n in nodes], dtype=float).reshape(-1, 3)
    n_gt_edges = len(gt["edges"])
    e = np.array([(row_of[x.source_id], row_of[x.target_id]) for x in edges], dtype=np.int64).reshape(-1, 2)
    if not len(nodes) or not len(e):
        return {"J": 0.0, "adj": 0.0, "n_pred": len(nodes), "n_est": gt["est_total"]}

    m = match_nodes(p_t, p_zyx * SCALE_ZYX, gt["t"], gt["zyx"] * SCALE_ZYX)
    e = np.unique(e, axis=0)                          # duplicate pairs count once
    e = e[p_t[e[:, 1]] - p_t[e[:, 0]] == 1]           # only dt == 1 edges are scored
    e = e[np.lexsort((e[:, 1], e[:, 0]))]
    rank = np.ones(len(e), np.int64)                  # out-degree capped at 2
    for i in range(1, len(e)):
        rank[i] = rank[i - 1] + 1 if e[i, 0] == e[i - 1, 0] else 1
    e = e[rank <= 2]

    ms, mt = m[e[:, 0]], m[e[:, 1]]
    gt_set = set(map(tuple, gt["edges"].tolist()))
    tp = sum((int(a), int(b)) in gt_set for a, b in zip(ms, mt) if a >= 0 and b >= 0)
    n_gt = len(gt["t"])
    gt_out = np.bincount(gt["edges"][:, 0], minlength=n_gt)
    gt_in = np.bincount(gt["edges"][:, 1], minlength=n_gt)
    graded = (np.where(ms >= 0, gt_out[np.clip(ms, 0, None)] > 0, False)
              | np.where(mt >= 0, gt_in[np.clip(mt, 0, None)] > 0, False))
    fp, fn = int(graded.sum()) - tp, n_gt_edges - tp
    j = tp / max(tp + fp + fn, 1)
    n_est = gt["est_total"]
    adj = max(0.0, j * (1 - NODE_ALPHA * (len(nodes) - n_est) / n_est)) if n_est else j
    return {"J": round(j, 4), "adj": round(adj, 4), "tp": tp, "fp": fp, "fn": fn,
            "n_pred": len(nodes), "n_est": n_est, "node_ratio": round(len(nodes) / max(n_est, 1), 3)}


In [ ]:
import time

SWEEP_THR_PCT = (85.0, 90.0, 95.0)   # 80 = current submission setting; 50-80 swept in v1
SAMPLES_PER_EMBRYO = 2

_test_stems = {p.stem for p in TEST_DIR.glob("*.zarr")}
_by_embryo: dict[str, list[Path]] = {}
for _zp in sorted(TRAIN_DIR.glob("*.zarr")):
    if _zp.stem not in _test_stems and _zp.with_suffix(".geff").exists():
        _by_embryo.setdefault(_zp.stem.split("_")[0], []).append(_zp)
sweep_paths = [p for paths in _by_embryo.values() for p in paths[:SAMPLES_PER_EMBRYO]]
print("sweep movies:", [p.stem for p in sweep_paths])

results = []
for zp in sweep_paths:
    gt = load_gt(zp)
    for thr in SWEEP_THR_PCT:
        # track_one_dataset calls detect_peaks_dog with its defaults, so swap the default threshold in.
        detect_peaks_dog.__defaults__ = (thr, None)
        t0 = time.time()
        nodes, edges, _ = track_one_dataset(zp)
        row = {"dataset": zp.stem, "thr_pct": thr, "sec": round(time.time() - t0),
               "n_edges": len(edges), **score_sample(nodes, edges, gt)}
        results.append(row)
        print(row, flush=True)
detect_peaks_dog.__defaults__ = (DOG_THR_PCT, None)

df = pd.DataFrame(results)
df.to_csv(OUTPUT_DIR / "thr_sweep.csv", index=False)
print(df.pivot(index="dataset", columns="thr_pct", values="adj"))
print("\nmean adj by threshold:\n", df.groupby("thr_pct")[["J", "adj", "node_ratio", "sec"]].mean())
